# OPRB: four lightweight OCR pipelines + Granite-Docling
Kaggle: Internet ON, T4 x2. LIMIT=5 is a smoke test; None selects all 9,000 pages. Recognition batching is not page batching. Tesseract uses CPU processes. Each model runs in a separate virtualenv. Download results before ending the session.


In [ ]:
from pathlib import Path
import json, os, subprocess, sys
FILES = {'pyproject.toml': '[build-system]\nrequires = ["setuptools>=68"]\nbuild-backend = "setuptools.build_meta"\n\n[project]\nname = "oprb-lab"\nversion = "0.1.0"\ndescription = "Pretrained OCR and VLM evaluation on occluded OPRB pages"\nrequires-python = ">=3.10"\ndependencies = ["Pillow>=10", "huggingface-hub>=0.30", "rapidfuzz>=3", "pandas>=2", "jinja2>=3", "matplotlib>=3.8", "tqdm>=4.66"]\n\n[project.optional-dependencies]\nvlm = ["transformers==4.57.1", "accelerate>=1,<2", "docling-core>=2.48,<3"]\npaddle = ["paddleocr==3.2.0"]\neasyocr = ["easyocr==1.7.2"]\nrapid = ["rapidocr-onnxruntime==1.4.4"]\ntest = ["pytest>=8"]\n\n[tool.setuptools.packages.find]\nwhere = ["src"]\n\n[tool.pytest.ini_options]\npythonpath = ["src"]\ntestpaths = ["tests"]\n', 'src/oprb_lab/__init__.py': '"""OPRB pretrained-model benchmark."""\n', 'src/oprb_lab/__main__.py': 'import argparse\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nfrom importlib.metadata import version, PackageNotFoundError\nfrom pathlib import Path\n\nfrom .data import collect, download\nfrom .models import MODELS, PROMPT\nfrom .data import file_sha\nfrom .report import summarize\nfrom .storage import fingerprint\n\n\ndef show_worker_logs(root, workers, tail=80):\n    """Expose startup/load failures hidden in redirected worker output."""\n    for rank, worker in enumerate(workers):\n        path = Path(root) / f"rank{rank}.log"\n        print(f"Worker {rank}: exit code {worker.returncode}; log: {path}", flush=True)\n        if path.exists():\n            from collections import deque\n            with path.open(encoding="utf-8", errors="replace") as stream:\n                print("".join(deque(stream, maxlen=tail)), flush=True)\n\n\ndef launch(args):\n    from huggingface_hub import model_info\n    samples = collect(args.data, args.limit, args.seed)\n    gpus = args.gpus.split(",") if args.model != "tesseract" else [""] * getattr(args, "cpu_workers", 2)\n    if args.model != "tesseract" and (any(not g.isdigit() for g in gpus) or len(set(gpus)) != len(gpus)):\n        raise ValueError("gpus must be unique numeric IDs, e.g. 0,1")\n    page_batch_size = getattr(args, "page_batch_size", 1)\n    if args.model != "granite" and page_batch_size != 1:\n        raise ValueError("Page batching is supported only for Granite; use recognition batching for OCR")\n    revision = model_info(MODELS[args.model], revision=args.revision).sha if args.model == "granite" else "package-pretrained-assets"\n    tessdata = getattr(args, "tessdata", None)\n    packages = {}\n    for name in ("torch", "torchvision", "transformers", "paddleocr", "paddlepaddle-gpu", "docling-core", "Pillow", "rapidfuzz", "easyocr", "rapidocr-onnxruntime", "onnxruntime-gpu"):\n        try:\n            packages[name] = version(name)\n        except PackageNotFoundError:\n            pass\n    config = dict(schema=3, model=args.model, model_id=MODELS[args.model], revision=revision, packages=packages,\n                  precision="float16" if args.model == "granite" else "pretrained engine default",\n                  max_tokens=args.max_tokens, rec_batch_size=getattr(args, "rec_batch_size", 16),\n                  page_batch_size=page_batch_size, tessdata=str(Path(tessdata).resolve()) if tessdata else None,\n                  tessdata_sha=file_sha(Path(tessdata) / "eng.traineddata") if args.model == "tesseract" and tessdata else None,\n                  execution="CPU processes" if args.model == "tesseract" else "one process per GPU",\n                  timing_mode="single page latency" if page_batch_size == 1 else "batch wall time / actual batch pages (throughput)",\n                  prompt=PROMPT if args.model == "granite" else None,\n                  limit=args.limit, seed=args.seed, world_size=len(gpus),\n                  manifest_sha=fingerprint([{k: v for k, v in s.items() if k not in ("image_path", "gt_path")} for s in samples]),\n                  normalization="NFKC whitespace casefold", gt_order="annotation order",\n                  code_sha=fingerprint({p.name: p.read_text(encoding="utf-8") for p in Path(__file__).parent.glob("*.py")}))\n    root = Path(args.output) / f"{args.model}-{fingerprint(config)}"\n    root.mkdir(parents=True, exist_ok=True)\n    (root / "config.json").write_text(json.dumps(config, indent=2))\n    (root / "manifest.json").write_text(json.dumps(samples, indent=2))\n    workers, logs = [], []\n    start = time.perf_counter()\n    try:\n        for rank, gpu in enumerate(gpus):\n            env = dict(os.environ, CUDA_VISIBLE_DEVICES=gpu, TOKENIZERS_PARALLELISM="false", OMP_NUM_THREADS="1",\n                       PYTHONHASHSEED=str(args.seed), CUBLAS_WORKSPACE_CONFIG=":4096:8")\n            log = (root / f"rank{rank}.log").open("a", encoding="utf-8")\n            logs.append(log)\n            workers.append(subprocess.Popen([sys.executable, "-u", "-m", "oprb_lab.worker", str(root), str(rank), str(len(gpus))],\n                                             env=env, stdout=log, stderr=subprocess.STDOUT))\n        print(f"Running {len(workers)} workers. Logs/results: {root}", flush=True)\n        while any(w.poll() is None for w in workers):\n            counts = {f"rank{rank}": sum(1 for _ in p.open(encoding="utf-8")) if p.exists() else 0\n                      for rank in range(len(workers)) for p in [root / f"rank{rank}.jsonl"]}\n            print(f"Saved rows: {counts}", flush=True)\n            time.sleep(10)\n    finally:\n        for worker in workers:\n            if worker.poll() is None:\n                worker.terminate()\n        for worker in workers:\n            worker.wait()\n        for log in logs:\n            log.close()\n    (root / "session.json").write_text(json.dumps(dict(wall_sec=time.perf_counter()-start,\n                                                        exit_codes=[w.returncode for w in workers]), indent=2))\n    summary = summarize(root)\n    if any(w.returncode for w in workers) or not summary["complete"]:\n        show_worker_logs(root, workers)\n        raise SystemExit("Incomplete evaluation: inspect logs; rerun the same command to retry errors/missing pages.")\n\n\ndef main():\n    parser = argparse.ArgumentParser(description="OPRB pretrained OCR benchmark")\n    sub = parser.add_subparsers(dest="command", required=True)\n    fetch = sub.add_parser("download")\n    fetch.add_argument("--data", default="data/oprb")\n    fetch.add_argument("--revision", default="main", help="Dataset revision; resolved to a commit before downloading")\n    report = sub.add_parser("summarize")\n    report.add_argument("run_dir")\n    run = sub.add_parser("run")\n    run.add_argument("--model", choices=MODELS, required=True)\n    run.add_argument("--data", default="data/oprb")\n    run.add_argument("--output", default="results")\n    run.add_argument("--gpus", default="0,1")\n    run.add_argument("--limit", type=int, default=None, help="Pages per category; omit for full test")\n    run.add_argument("--seed", type=int, default=42)\n    run.add_argument("--revision", default="main")\n    run.add_argument("--max-tokens", type=int, default=4096)\n    run.add_argument("--max-pixels", type=int, default=1280*28*28)\n    run.add_argument("--max-tiles", type=int, default=12)\n    run.add_argument("--rec-batch-size", type=int, default=16, help="OCR line-crop recognition batch size")\n    run.add_argument("--page-batch-size", type=int, default=1, help="Granite pages per batch; >1 measures amortized throughput")\n    run.add_argument("--cpu-workers", type=int, default=2, help="Tesseract CPU process count")\n    run.add_argument("--tessdata", help="Directory containing tessdata_fast/eng.traineddata")\n    args = parser.parse_args()\n    if args.command == "download":\n        download(args.data, args.revision)\n    elif args.command == "summarize":\n        summarize(args.run_dir)\n    else:\n        if min(args.max_tokens, args.max_pixels, args.max_tiles, args.rec_batch_size, args.page_batch_size, args.cpu_workers) < 1:\n            parser.error("Token, pixel, and tile limits must be positive")\n        if args.model == "tesseract" and not args.tessdata:\n            parser.error("Tesseract requires --tessdata pointing to tessdata_fast/eng.traineddata")\n        launch(args)\n\n\nif __name__ == "__main__":\n    main()\n', 'src/oprb_lab/comparison.py': '"""Compare the five registered models on matched category-qualified pages."""\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom .models import MODELS\nfrom .report import METRICS\nfrom .storage import read_rows\n\n\ndef compare(run_dirs, output):\n    import pandas as pd\n    runs, manifests, configs, frames = {}, [], [], []\n    for directory in map(Path, run_dirs):\n        config = json.loads((directory / "config.json").read_text())\n        key = config["model"]\n        if key not in MODELS or key in runs:\n            raise ValueError("Supply exactly one run per registered model")\n        runs[key] = directory\n        configs.append(config)\n        manifest = json.loads((directory / "manifest.json").read_text())\n        expected = {s["uid"] for s in manifest}\n        if len(expected) != len(manifest):\n            raise ValueError("Duplicate manifest UIDs")\n        manifests.append(expected)\n        rows = {}\n        for path in sorted(directory.glob("rank[0-9]*.jsonl")):\n            for row in read_rows(path):\n                if row["uid"] in expected:\n                    rows[row["uid"]] = dict(row, model=key)\n        frames.append(pd.DataFrame(rows.values(), columns=None if rows else ["uid", "status", "model"]))\n    if set(runs) != set(MODELS):\n        raise ValueError(f"All five models are required: {list(MODELS)}")\n    if any(ids != manifests[0] for ids in manifests):\n        raise ValueError("Runs use different sample manifests")\n    for field in ("manifest_sha", "normalization", "gt_order", "seed", "code_sha"):\n        if any(c.get(field) != configs[0].get(field) for c in configs):\n            raise ValueError(f"Incompatible evaluation protocol: {field}")\n    data = pd.concat(frames, ignore_index=True)\n    coverage = []\n    common = set(manifests[0])\n    for key in MODELS:\n        subset = data[data.model == key]\n        successful = set(subset.loc[subset.status == "ok", "uid"])\n        common &= successful\n        coverage.append(dict(model=key, expected=len(manifests[0]), success=len(successful),\n                             errors=int((subset.status == "error").sum()),\n                             missing=len(manifests[0] - set(subset.uid))))\n    output = Path(output)\n    output.mkdir(parents=True, exist_ok=True)\n    pd.DataFrame(coverage).to_csv(output / "coverage.csv", index=False)\n    if not common:\n        raise ValueError("No common successful pages; inspect coverage.csv")\n    paired = data[(data.status == "ok") & data.uid.isin(common)].copy()\n    # Comparing GT also catches inconsistent annotations despite equal sample names.\n    if paired.groupby("uid").ground_truth.nunique(dropna=False).gt(1).any():\n        raise ValueError("Ground truth differs across runs")\n    table = paired.groupby("model")[METRICS].mean()\n    table["n_common"] = len(common)\n    modes = {config.get("timing_mode", "single page latency") for config in configs}\n    table["pareto"] = [not any((other.latency_sec <= row.latency_sec and other.wer <= row.wer)\n                       and (other.latency_sec < row.latency_sec or other.wer < row.wer)\n                       for _, other in table.iterrows()) for _, row in table.iterrows()] if len(modes) == 1 else None\n    settings = {config["model"]: config for config in configs}\n    for field in ("execution", "world_size", "rec_batch_size", "page_batch_size", "timing_mode"):\n        table[field] = [settings[key].get(field) for key in table.index]\n    table.to_csv(output / "comparison.csv")\n    (output / "comparison.tex").write_text(table.to_latex(float_format="%.4f"), encoding="utf-8")\n    paired.to_csv(output / "paired_pages.csv", index=False)\n    for group in ("category", "occlusion_type", "density"):\n        paired.groupby(["model", group])[METRICS].mean().to_csv(output / f"by_{group}.csv")\n    import matplotlib\n    matplotlib.use("Agg")\n    import matplotlib.pyplot as plt\n    fig, ax = plt.subplots()\n    for key, row in table.iterrows():\n        ax.scatter(row.latency_sec, row.wer)\n        ax.annotate(key, (row.latency_sec, row.wer))\n    ax.set(xlabel="Mean end-to-end latency (s/page)", ylabel="Macro WER", title="Matched-page comparison")\n    fig.tight_layout()\n    fig.savefig(output / "latency_wer.png", dpi=180)\n    plt.close(fig)\n    (output / "protocol.json").write_text(json.dumps(dict(\n        runs={k: str(v) for k, v in runs.items()}, configs=configs,\n        common_uids=sorted(common), note="Report coverage alongside paired scores. Tesseract is CPU-only. Timings are deployment-specific; Pareto omitted for mixed latency/throughput modes."), indent=2))\n    return table\n\n\nif __name__ == "__main__":\n    parser = argparse.ArgumentParser()\n    parser.add_argument("runs", nargs="+", help="Exactly five fingerprinted run directories")\n    parser.add_argument("--output", default="comparison")\n    args = parser.parse_args()\n    print(compare(args.runs, args.output))\n', 'src/oprb_lab/data.py': '"""Download only occluded test images and scoring annotations."""\nimport random\nimport hashlib\nimport shutil\nimport tarfile\nfrom pathlib import Path\n\nEXPECTED = {"text": 6000, "nontext": 3000}\n\n\ndef extract_flat(archive, destination, suffix):\n    destination = Path(destination)\n    destination.mkdir(parents=True, exist_ok=True)\n    seen = set()\n    with tarfile.open(archive, "r:gz") as tar:\n        for item in tar:\n            if not item.isfile() or not item.name.lower().endswith(suffix):\n                continue\n            name = item.name.replace("\\\\", "/").split("/")[-1]\n            if name in seen:\n                raise ValueError(f"Duplicate archive basename: {name}")\n            seen.add(name)\n            with tar.extractfile(item) as source, (destination / name).open("wb") as target:\n                shutil.copyfileobj(source, target)\n\n\ndef download(root, revision="main"):\n    from huggingface_hub import hf_hub_download, dataset_info\n    revision = dataset_info("kpurkayastha/OPRB", revision=revision).sha\n    for category in EXPECTED:\n        for kind, suffix in (("img", ".jpg"), ("gt", ".txt")):\n            destination = Path(root) / category / kind\n            marker = destination / ".complete"\n            if marker.exists() and marker.read_text() == revision and len(list(destination.glob(f"*{suffix}"))) == EXPECTED[category]:\n                continue\n            archive = hf_hub_download("kpurkayastha/OPRB", f"test/{category}/{kind}.tar.gz", repo_type="dataset", revision=revision)\n            extract_flat(archive, destination, suffix)\n            count = len(list(destination.glob(f"*{suffix}")))\n            if count != EXPECTED[category]:\n                raise ValueError(f"{category}/{kind}: expected {EXPECTED[category]}, got {count}")\n            marker.write_text(revision)\n\n\ndef file_sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for block in iter(lambda: stream.read(1024 * 1024), b""):\n            digest.update(block)\n    return digest.hexdigest()\n\n\ndef collect(root, limit=None, seed=42):\n    if limit is not None and limit < 1:\n        raise ValueError("limit must be positive or None")\n    samples = []\n    for category in EXPECTED:\n        images = {p.stem: p for p in (Path(root) / category / "img").glob("*.jpg")}\n        labels = {p.stem: p for p in (Path(root) / category / "gt").glob("*.txt")}\n        if not images or images.keys() != labels.keys():\n            raise ValueError(f"Missing or unmatched images/GT in {category}")\n        if limit is None and len(images) != EXPECTED[category]:\n            raise ValueError(f"Full test requires {EXPECTED[category]} pairs in {category}; got {len(images)}")\n        keys = sorted(images)\n        if limit is not None:\n            keys = sorted(random.Random(seed).sample(keys, min(limit, len(keys))))\n        for key in keys:\n            parts = key.split("__", 2)\n            samples.append(dict(uid=f"{category}/{key}", sample_id=key, category=category,\n                                image_path=str(images[key].resolve()), gt_path=str(labels[key].resolve()),\n                                image_sha=file_sha(images[key]), gt_sha=file_sha(labels[key]),\n                                occlusion_type=parts[0], density=parts[1] if len(parts) > 1 else "unknown"))\n    return samples\n\n\ndef read_gt(path):\n    # Keep annotation order: this policy must be disclosed for multi-column pages.\n    tokens = []\n    for line in Path(path).read_text(encoding="utf-8-sig").splitlines():\n        fields = line.split("\\t")\n        if len(fields) >= 10 and fields[0].strip():\n            tokens.append(fields[0].strip())\n    if not tokens:\n        raise ValueError(f"No GT tokens: {path}")\n    return " ".join(tokens)\n\n\ndef shard(samples, rank, world_size):\n    if world_size < 1 or not 0 <= rank < world_size:\n        raise ValueError("Invalid rank/world_size")\n    return samples[rank::world_size]\n', 'src/oprb_lab/metrics.py': 'import re\nimport unicodedata\nfrom collections import Counter\n\n\ndef normalize(text):\n    return re.sub(r"\\s+", " ", unicodedata.normalize("NFKC", text)).strip().casefold()\n\n\ndef score(reference, prediction):\n    from rapidfuzz.distance import Levenshtein\n    ref, hyp = normalize(reference), normalize(prediction)\n    if not ref:\n        raise ValueError("Empty reference")\n    a, b = Counter(ref.split()), Counter(hyp.split())\n    matched = sum((a & b).values())\n    precision = matched / sum(b.values()) if b else 0.0\n    recall = matched / sum(a.values())\n    return dict(cer=Levenshtein.distance(ref, hyp) / len(ref),\n                wer=Levenshtein.distance(ref.split(), hyp.split()) / len(ref.split()),\n                token_precision=precision, token_recall=recall,\n                token_f1=2 * precision * recall / (precision + recall) if precision + recall else 0.0)\n', 'src/oprb_lab/models.py': '"""Five image-only pretrained OCR adapters. GT is never passed here."""\nimport hashlib\nimport os\nimport shutil\nimport subprocess\nfrom pathlib import Path\n\nfrom PIL import Image\n\nMODELS = {\n    "paddle": "PP-OCRv5_mobile_det+PP-OCRv5_mobile_rec",\n    "rapid": "PP-OCRv4-Mobile-ONNX (rapidocr_onnxruntime-1.4.4)",\n    "easyocr": "EasyOCR-English-CRAFT+CRNN",\n    "tesseract": "Tesseract-eng-tessdata_fast",\n    "granite": "ibm-granite/granite-docling-258M",\n}\nPROMPT = "Convert this page to docling."\nTORCH_MODELS = {"easyocr", "granite"}\n\n\ndef output_text(text, raw=None, tokens=None, truncated=False):\n    return dict(prediction=text, raw_prediction=text if raw is None else raw,\n                generated_tokens=tokens, possibly_truncated=truncated)\n\n\ndef sha(path):\n    digest = hashlib.sha256()\n    with Path(path).open("rb") as stream:\n        for chunk in iter(lambda: stream.read(1048576), b""):\n            digest.update(chunk)\n    return digest.hexdigest()\n\n\ndef ort_session(component):\n    """Find the actual ORT session in the pinned RapidOCR wrapper."""\n    if hasattr(component, "get_providers"):\n        return component\n    for attr in ("session", "infer", "predictor"):\n        child = getattr(component, attr, None)\n        if child is not None and child is not component:\n            try:\n                return ort_session(child)\n            except ValueError:\n                pass\n    raise ValueError("Cannot verify RapidOCR CUDA session; refusing an unverified GPU run")\n\n\nclass Adapter:\n    def __init__(self, name, revision=None, max_tokens=4096, max_pixels=None, max_tiles=None,\n                 rec_batch_size=16, tessdata=None):\n        if name not in MODELS:\n            raise ValueError(f"Unknown model: {name}")\n        self.name, self.max_tokens, self.rec_batch_size = name, max_tokens, rec_batch_size\n        self.parameter_count = None\n        self.metadata = {}\n        if name == "paddle":\n            from paddleocr import PaddleOCR\n            self.model = PaddleOCR(\n                text_detection_model_name="PP-OCRv5_mobile_det", text_recognition_model_name="PP-OCRv5_mobile_rec",\n                use_doc_orientation_classify=False, use_doc_unwarping=False, use_textline_orientation=False,\n                text_recognition_batch_size=rec_batch_size, device="gpu:0")\n            cache = Path.home() / ".paddlex" / "official_models"\n            self.metadata["weight_hashes"] = {\n                str(path.relative_to(cache)): sha(path)\n                for model in ("PP-OCRv5_mobile_det", "PP-OCRv5_mobile_rec")\n                for path in (cache / model).glob("*.pdiparams")}\n            return\n        if name == "rapid":\n            import onnxruntime as ort\n            ort.preload_dlls(directory="")\n            import rapidocr_onnxruntime as package\n            from rapidocr_onnxruntime import RapidOCR\n            model_dir = Path(package.__file__).parent / "models"\n            det, rec = model_dir / "ch_PP-OCRv4_det_infer.onnx", model_dir / "ch_PP-OCRv4_rec_infer.onnx"\n            if not det.exists() or not rec.exists():\n                raise FileNotFoundError("Pinned PP-OCRv4 weights missing from RapidOCR package")\n            self.model = RapidOCR(det_model_path=str(det), rec_model_path=str(rec),\n                                  det_use_cuda=True, rec_use_cuda=True, cls_use_cuda=False,\n                                  rec_batch_num=rec_batch_size, intra_op_num_threads=1, inter_op_num_threads=1)\n            providers = {key: ort_session(getattr(self.model, key)).get_providers() for key in ("text_det", "text_rec")}\n            if any("CUDAExecutionProvider" not in values for values in providers.values()):\n                raise RuntimeError(f"RapidOCR GPU provider failed; refusing silent CPU fallback: {providers}")\n            self.metadata = dict(providers=providers, det_sha=sha(det), rec_sha=sha(rec),\n                                 language="bundled Chinese/English PP-OCRv4 dictionary", classifier=False)\n            return\n        if name == "tesseract":\n            self.binary = shutil.which("tesseract")\n            if not self.binary:\n                raise RuntimeError("Install the tesseract-ocr system binary first")\n            self.tessdata = Path(tessdata or os.environ.get("OPRB_TESSDATA", ""))\n            weights = self.tessdata / "eng.traineddata"\n            if not weights.exists():\n                raise FileNotFoundError("Provide --tessdata containing tessdata_fast/eng.traineddata")\n            self.metadata = dict(tesseract=subprocess.check_output([self.binary, "--version"], text=True).splitlines()[0],\n                                 weights_sha=sha(weights), language="eng", oem=1, psm=3)\n            return\n        import torch\n        self.torch, self.device, self.dtype = torch, "cuda:0", torch.float16\n        if name == "easyocr":\n            import easyocr\n            self.model = easyocr.Reader(["en"], gpu=self.device, recog_network="standard", verbose=False)\n            self.parameter_count = sum(p.numel() for module in (self.model.detector, self.model.recognizer) for p in module.parameters())\n            self.metadata = dict(language="en", detector="CRAFT", recognizer="english_g2")\n            cache = getattr(self.model, "model_storage_directory", None)\n            if cache:\n                self.metadata["weight_hashes"] = {path.name: sha(path) for path in Path(cache).glob("*.pth")}\n            return\n        from transformers import AutoProcessor, AutoModelForImageTextToText\n        self.processor = AutoProcessor.from_pretrained(MODELS[name], revision=revision)\n        self.processor.tokenizer.padding_side = "left"\n        self.model = AutoModelForImageTextToText.from_pretrained(MODELS[name], revision=revision,\n                    torch_dtype=self.dtype, low_cpu_mem_usage=True, attn_implementation="sdpa").to(self.device).eval()\n        self.parameter_count = sum(p.numel() for p in self.model.parameters())\n\n    def predict(self, path):\n        if self.name == "paddle":\n            text = "\\n".join(t for result in self.model.predict(str(path)) for t in result["rec_texts"])\n            return output_text(text)\n        if self.name == "rapid":\n            rows, _ = self.model(str(path), use_cls=False)\n            return output_text("\\n".join(row[1] for row in (rows or [])))\n        if self.name == "easyocr":\n            rows = self.model.readtext(str(path), batch_size=self.rec_batch_size, detail=1, paragraph=False, workers=0)\n            return output_text("\\n".join(row[1] for row in rows))\n        if self.name == "tesseract":\n            text = subprocess.check_output([self.binary, str(path), "stdout", "--tessdata-dir", str(self.tessdata),\n                                           "-l", "eng", "--oem", "1", "--psm", "3"], text=True, encoding="utf-8", timeout=180)\n            return output_text(text.strip())\n        return self.predict_batch([path])[0]\n\n    def predict_batch(self, paths):\n        if self.name != "granite":\n            return [self.predict(path) for path in paths]\n        from docling_core.types.doc import DoclingDocument\n        from docling_core.types.doc.document import DocTagsDocument\n        images = []\n        for path in paths:\n            with Image.open(path) as source:\n                images.append(source.convert("RGB"))\n        messages = [{"role": "user", "content": [{"type": "image"}, {"type": "text", "text": PROMPT}]}]\n        prompt = self.processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)\n        inputs = self.processor(text=[prompt] * len(images), images=images, padding=True, return_tensors="pt").to(self.device)\n        for key, tensor in inputs.items():\n            if tensor.is_floating_point():\n                inputs[key] = tensor.to(self.dtype)\n        with self.torch.inference_mode():\n            generated = self.model.generate(**inputs, max_new_tokens=self.max_tokens, do_sample=False)\n        answers = []\n        for ids, image in zip(generated[:, inputs["input_ids"].shape[-1]:], images):\n            eos = self.model.generation_config.eos_token_id\n            eos = eos if isinstance(eos, list) else [eos]\n            tokens = ids.tolist()\n            stop = next((i + 1 for i, token in enumerate(tokens) if token in eos), len(tokens))\n            raw = self.processor.decode(ids[:stop], skip_special_tokens=False, clean_up_tokenization_spaces=False)\n            doc = DoclingDocument.load_from_doctags(DocTagsDocument.from_doctags_and_image_pairs([raw], [image]), document_name="page")\n            answers.append(output_text(doc.export_to_text(), raw, stop, stop >= self.max_tokens))\n        return answers\n', 'src/oprb_lab/report.py': 'import json\nfrom pathlib import Path\n\nfrom .storage import read_rows\n\nMETRICS = ["cer", "wer", "token_precision", "token_recall", "token_f1", "latency_sec"]\n\n\ndef summarize(run_dir):\n    import pandas as pd\n    root = Path(run_dir)\n    manifest = json.loads((root / "manifest.json").read_text())\n    expected = {s["uid"] for s in manifest}\n    rows = {}\n    for path in sorted(root.glob("rank[0-9]*.jsonl")):\n        for row in read_rows(path):\n            if row["uid"] in expected:\n                rows[row["uid"]] = row\n    df = pd.DataFrame(rows.values()) if rows else pd.DataFrame(columns=["uid", "status"])\n    df.to_csv(root / "per_image_results.csv", index=False)\n    ok = df.loc[df.status == "ok"]\n    summary = dict(expected=len(expected), observed=len(df), success=len(ok),\n                   errors=int((df.status == "error").sum()), missing=len(expected - rows.keys()),\n                   complete=len(ok) == len(expected), aggregation="macro mean over successful pages")\n    if len(ok):\n        summary.update({m: float(ok[m].mean()) for m in METRICS})\n        summary["possibly_truncated"] = int(ok.possibly_truncated.fillna(False).sum())\n        summary["truncation_unknown"] = int(ok.possibly_truncated.isna().sum())\n        for groups, name in [(["category"], "category"), (["category", "occlusion_type"], "type"),\n                             (["category", "density"], "density")]:\n            ok.groupby(groups, dropna=False).agg(n=("uid", "count"), **{m: (m, "mean") for m in METRICS}).reset_index().to_csv(\n                root / f"summary_by_{name}.csv", index=False)\n        import matplotlib\n        matplotlib.use("Agg")\n        import matplotlib.pyplot as plt\n        ax = ok.groupby("category")[["cer", "wer"]].mean().plot.bar(rot=0)\n        ax.set_ylabel("Macro error rate (lower is better)")\n        plt.tight_layout()\n        plt.savefig(root / "error_rates.png", dpi=160)\n        plt.close()\n    (root / "summary.json").write_text(json.dumps(summary, indent=2))\n    print(json.dumps(summary, indent=2))\n    return summary\n', 'src/oprb_lab/storage.py': 'import hashlib\nimport json\nfrom pathlib import Path\n\n\ndef fingerprint(config):\n    return hashlib.sha256(json.dumps(config, sort_keys=True).encode()).hexdigest()[:16]\n\n\ndef read_rows(path):\n    path = Path(path)\n    if not path.exists():\n        return []\n    lines = path.read_text(encoding="utf-8").splitlines()\n    rows = []\n    for i, line in enumerate(lines):\n        try:\n            rows.append(json.loads(line))\n        except json.JSONDecodeError:\n            if i != len(lines) - 1:\n                raise\n            # Interrupted append: caller rewrites valid records before resuming.\n    return rows\n\n\ndef rewrite_rows(path, rows):\n    path = Path(path)\n    tmp = path.with_suffix(".tmp")\n    tmp.write_text("".join(json.dumps(r, ensure_ascii=False) + "\\n" for r in rows), encoding="utf-8")\n    tmp.replace(path)\n\n\ndef resume_rows(path, selected):\n    """Keep only successful rows for this shard; errors remain eligible for retry."""\n    ids = {sample["uid"] for sample in selected}\n    completed = {row["uid"]: row for row in read_rows(path)\n                 if row.get("status") == "ok" and row["uid"] in ids}\n    rewrite_rows(path, list(completed.values()))\n    return [sample for sample in selected if sample["uid"] not in completed]\n', 'src/oprb_lab/worker.py': '"""Shard pages over independent GPU processes, or CPU processes for Tesseract."""\nimport argparse\nimport json\nimport os\nimport platform\nimport random\nimport time\nfrom importlib.metadata import version\nfrom pathlib import Path\n\nfrom .data import read_gt, shard\nfrom .metrics import score\nfrom .models import Adapter, TORCH_MODELS\nfrom .storage import resume_rows\n\n\ndef run(run_dir, rank, world_size):\n    root = Path(run_dir)\n    config = json.loads((root / \'config.json\').read_text())\n    name = config[\'model\']\n    random.seed(config[\'seed\'])\n    import numpy as np\n    np.random.seed(config[\'seed\'])\n    torch = None\n    if name == \'paddle\':\n        import paddle\n        paddle.seed(config[\'seed\'])\n    elif name in TORCH_MODELS:\n        import torch\n        torch.manual_seed(config[\'seed\'])\n        torch.cuda.manual_seed_all(config[\'seed\'])\n        torch.backends.cudnn.benchmark = False\n        torch.backends.cudnn.deterministic = True\n        torch.use_deterministic_algorithms(True, warn_only=True)\n    samples = json.loads((root / \'manifest.json\').read_text())\n    selected = shard(samples, rank, world_size)\n    path = root / f\'rank{rank}.jsonl\'\n    todo = resume_rows(path, selected)\n    if not todo:\n        return\n    model = Adapter(name, config[\'revision\'], config[\'max_tokens\'],\n                    rec_batch_size=config.get(\'rec_batch_size\', 16), tessdata=config.get(\'tessdata\'))\n    def synchronize():\n        if name == \'paddle\':\n            paddle.device.cuda.synchronize()\n        elif torch is not None:\n            torch.cuda.synchronize()\n        # ONNX run() and the Tesseract subprocess return completed CPU text.\n    model.predict(todo[0][\'image_path\'])\n    synchronize()\n    metadata = dict(rank=rank, physical_gpu=os.environ.get(\'CUDA_VISIBLE_DEVICES\'),\n                    parameter_count=model.parameter_count, python=platform.python_version(),\n                    platform=platform.platform(), adapter=getattr(model, \'metadata\', {}))\n    if torch is not None:\n        metadata.update(gpu=torch.cuda.get_device_name(0), torch=version(\'torch\'), cuda=torch.version.cuda)\n    elif name == \'paddle\':\n        metadata.update(gpu=paddle.device.cuda.get_device_name(0), paddle=paddle.__version__,\n                        cuda=paddle.version.cuda(), paddleocr=version(\'paddleocr\'))\n    elif name == \'rapid\':\n        metadata.update(onnxruntime=version(\'onnxruntime-gpu\'), gpu=\'CUDAExecutionProvider; worker GPU mask\')\n    else:\n        metadata.update(gpu=None, device=\'CPU\')\n    (root / f\'rank{rank}_metadata.json\').write_text(json.dumps(metadata, indent=2))\n    batch_size = config.get(\'page_batch_size\', 1)\n    for offset in range(0, len(todo), batch_size):\n        batch = todo[offset:offset+batch_size]\n        try:\n            synchronize()\n            start = time.perf_counter()\n            outputs = model.predict_batch([sample[\'image_path\'] for sample in batch]) if batch_size > 1 else [model.predict(batch[0][\'image_path\'])]\n            synchronize()\n            elapsed = time.perf_counter() - start\n            if len(outputs) != len(batch):\n                raise ValueError(\'Adapter output count differs from batch size\')\n            batch_error = None\n        except Exception as error:\n            outputs, elapsed, batch_error = [None] * len(batch), None, repr(error)\n            if torch is not None and isinstance(error, torch.cuda.OutOfMemoryError):\n                torch.cuda.empty_cache()\n        for index, (sample, output) in enumerate(zip(batch, outputs), start=offset+1):\n            row = dict(sample)\n            try:\n                if batch_error:\n                    raise RuntimeError(batch_error)\n                gt = read_gt(sample[\'gt_path\'])\n                row.update(status=\'ok\', latency_sec=elapsed/len(batch), batch_wall_sec=elapsed,\n                           actual_page_batch=len(batch), timing_mode=config.get(\'timing_mode\', \'single page latency\'),\n                           ground_truth=gt, **output, **score(gt, output[\'prediction\']))\n            except Exception as error:\n                row.update(status=\'error\', error=repr(error))\n                print(f"rank={rank} ERROR {sample[\'uid\']}: {error}", flush=True)\n            with path.open(\'a\', encoding=\'utf-8\') as stream:\n                stream.write(json.dumps(row, ensure_ascii=False) + \'\\n\')\n                stream.flush()\n                os.fsync(stream.fileno())\n            print(f"rank={rank} {index}/{len(todo)} {row[\'status\']}", flush=True)\n\n\nif __name__ == \'__main__\':\n    parser = argparse.ArgumentParser()\n    parser.add_argument(\'run_dir\')\n    parser.add_argument(\'rank\', type=int)\n    parser.add_argument(\'world_size\', type=int)\n    args = parser.parse_args()\n    run(args.run_dir, args.rank, args.world_size)\n', 'tests/test_adapters.py': '"""Offline adapter regression tests; real GPU tests run separately on Kaggle."""\nimport sys\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport pytest\n\nfrom oprb_lab.models import Adapter, MODELS, ort_session\n\n\ndef test_registry_is_exact_new_five():\n    assert set(MODELS) == {\'paddle\', \'rapid\', \'easyocr\', \'tesseract\', \'granite\'}\n    with pytest.raises(ValueError):\n        Adapter(\'qwen\')\n\n\ndef test_paddle_passes_recognition_batch(monkeypatch):\n    params = {}\n    class OCR:\n        def __init__(self, **kw): params.update(kw)\n        def predict(self, path): return [{\'rec_texts\':[\'one\',\'two\']}]\n    monkeypatch.setitem(sys.modules,\'paddleocr\',SimpleNamespace(PaddleOCR=OCR))\n    adapter=Adapter(\'paddle\',rec_batch_size=32)\n    assert params[\'text_recognition_batch_size\']==32\n    assert params[\'text_detection_model_name\']==\'PP-OCRv5_mobile_det\'\n    assert adapter.predict(\'x\')[\'prediction\']==\'one\\ntwo\'\n\n\ndef test_rapid_verifies_real_sessions_and_pinned_weights(tmp_path,monkeypatch):\n    models=tmp_path/\'models\'\n    models.mkdir()\n    for name in (\'ch_PP-OCRv4_det_infer.onnx\',\'ch_PP-OCRv4_rec_infer.onnx\'):\n        (models/name).write_bytes(b\'weights\')\n    providers=[\'CUDAExecutionProvider\',\'CPUExecutionProvider\']\n    session=SimpleNamespace(get_providers=lambda:providers)\n    params={}\n    class Rapid:\n        def __init__(self,**kw):\n            params.update(kw)\n            self.text_det=SimpleNamespace(infer=SimpleNamespace(session=session))\n            self.text_rec=SimpleNamespace(session=SimpleNamespace(session=session))\n        def __call__(self,path,**kw):\n            assert kw[\'use_cls\'] is False\n            return [[[], \'cat\', .9]],None\n    monkeypatch.setitem(sys.modules,\'onnxruntime\',SimpleNamespace(preload_dlls=lambda **kw:None))\n    monkeypatch.setitem(sys.modules,\'rapidocr_onnxruntime\',SimpleNamespace(__file__=str(tmp_path/\'__init__.py\'),RapidOCR=Rapid))\n    a=Adapter(\'rapid\',rec_batch_size=16)\n    assert a.predict(\'x\')[\'prediction\']==\'cat\'\n    assert params[\'det_use_cuda\'] and params[\'rec_use_cuda\']\n    providers[:]=[\'CPUExecutionProvider\']\n    with pytest.raises(RuntimeError,match=\'fallback\'):\n        Adapter(\'rapid\')\n\n\ndef test_ort_unknown_wrapper_fails():\n    with pytest.raises(ValueError): ort_session(object())\n\n\ndef test_tesseract_uses_explicit_fast_weights_and_timeout(tmp_path,monkeypatch):\n    from oprb_lab import models\n    (tmp_path/\'eng.traineddata\').write_bytes(b\'fast\')\n    monkeypatch.setattr(models.shutil,\'which\',lambda name:\'/usr/bin/tesseract\')\n    commands=[]\n    def execute(cmd,**kw):\n        commands.append((cmd,kw))\n        return \'tesseract 5.5\' if \'--version\' in cmd else \'some text\\n\'\n    monkeypatch.setattr(models.subprocess,\'check_output\',execute)\n    a=Adapter(\'tesseract\',tessdata=tmp_path)\n    assert a.predict(\'page.jpg\')[\'prediction\']==\'some text\'\n    command,options=commands[-1]\n    assert command[command.index(\'--tessdata-dir\')+1]==str(tmp_path)\n    assert command[command.index(\'--psm\')+1]==\'3\'\n    assert options[\'timeout\']==180\n    assert a.metadata[\'weights_sha\']\n', 'tests/test_comparison.py': 'import json\n\nimport pytest\n\nfrom oprb_lab.comparison import compare\nfrom oprb_lab.metrics import score\nfrom oprb_lab.models import MODELS\nfrom oprb_lab.storage import rewrite_rows, resume_rows, read_rows\n\n\ndef make_runs(tmp_path):\n    runs = []\n    for key in MODELS:\n        path = tmp_path / key\n        path.mkdir()\n        (path / "config.json").write_text(json.dumps(dict(model=key, manifest_sha="same", normalization="nfkc", seed=42, code_sha="same", gt_order="annotation")))\n        (path / "manifest.json").write_text(json.dumps([dict(uid="text/a"), dict(uid="nontext/a")]))\n        rows = [dict(uid=f"{cat}/a", category=cat, status="ok", ground_truth="cat", prediction="cat",\n                     occlusion_type="Ink", density="2", latency_sec=1., **score("cat", "cat")) for cat in ("text", "nontext")]\n        rewrite_rows(path / "rank0.jsonl", rows)\n        runs.append(path)\n    return runs\n\n\ndef test_comparison_uses_common_success_and_reports_failure(tmp_path):\n    runs = make_runs(tmp_path)\n    rows = read_rows(runs[0] / "rank0.jsonl")\n    rows[1] = dict(uid="nontext/a", status="error", error="OOM")\n    rewrite_rows(runs[0] / "rank0.jsonl", rows)\n    table = compare(runs, tmp_path / "output")\n    assert set(table.index) == set(MODELS)\n    assert (table.n_common == 1).all()\n    import pandas as pd\n    coverage = pd.read_csv(tmp_path / "output/coverage.csv").set_index("model")\n    assert coverage.loc["paddle", "errors"] == 1\n    assert coverage.loc["paddle", "expected"] == 2\n\n\n@pytest.mark.parametrize("field", ["manifest_sha", "normalization", "code_sha", "seed"])\ndef test_comparison_rejects_changed_protocol(tmp_path, field):\n    runs = make_runs(tmp_path)\n    path = runs[0] / "config.json"\n    config = json.loads(path.read_text())\n    config[field] = "changed"\n    path.write_text(json.dumps(config))\n    with pytest.raises(ValueError, match="protocol"):\n        compare(runs, tmp_path / "output")\n\n\ndef test_comparison_requires_all_five(tmp_path):\n    with pytest.raises(ValueError, match="All five"):\n        compare(make_runs(tmp_path)[:4], tmp_path / "output")\n\n\ndef test_comparison_rejects_inconsistent_annotations(tmp_path):\n    runs = make_runs(tmp_path)\n    path = runs[0] / "rank0.jsonl"\n    rows = read_rows(path)\n    rows[0]["ground_truth"] = "different"\n    rewrite_rows(path, rows)\n    with pytest.raises(ValueError, match="Ground truth"):\n        compare(runs, tmp_path / "output")\n\n\ndef test_comparison_no_common_success_writes_coverage(tmp_path):\n    runs = make_runs(tmp_path)\n    rewrite_rows(runs[0] / "rank0.jsonl", [])\n    with pytest.raises(ValueError, match="No common"):\n        compare(runs, tmp_path / "output")\n    assert (tmp_path / "output/coverage.csv").exists()\n\n\ndef test_resume_retries_errors_and_qualifies_category(tmp_path):\n    path = tmp_path / "rank0.jsonl"\n    selected = [dict(uid="text/a"), dict(uid="nontext/a")]\n    rewrite_rows(path, [dict(uid="text/a", status="ok"), dict(uid="nontext/a", status="error"), dict(uid="other", status="ok")])\n    assert resume_rows(path, selected) == [dict(uid="nontext/a")]\n    assert read_rows(path) == [dict(uid="text/a", status="ok")]\n', 'tests/test_core.py': 'import io\nimport json\nimport tarfile\n\nimport pytest\nfrom PIL import Image\n\nfrom oprb_lab.data import collect, extract_flat, read_gt, shard\nfrom oprb_lab.metrics import normalize, score\nfrom oprb_lab.storage import fingerprint, read_rows, rewrite_rows\n\n\ndef test_metrics_duplicates_and_insertions():\n    result = score("cat cat dog", "cat dog dog dog")\n    assert result["token_precision"] == .5\n    assert result["token_recall"] == pytest.approx(2/3)\n    assert score("a", "a b c")["wer"] == 2\n    assert score("abc", "")["cer"] == 1\n    assert score("abc", "abc")["cer"] == 0\n    assert score("The cat sat", "The cat")["wer"] == pytest.approx(1/3)\n    with pytest.raises(ValueError):\n        score("", "abc")\n\n\ndef test_normalization():\n    assert normalize("  Ａ\\nB\\t c ") == "a b c"\n\n\ndef test_gt_accepts_ten_column_annotations(tmp_path):\n    path = tmp_path / "page.txt"\n    path.write_text("word\\t" + "\\t".join(["0"] * 9), encoding="utf-8")\n    assert read_gt(path) == "word"\n\n\ndef test_shards_are_disjoint_complete():\n    samples = list(range(13))\n    shards = [shard(samples, i, 2) for i in range(2)]\n    assert set(shards[0]).isdisjoint(shards[1])\n    assert sorted(shards[0] + shards[1]) == samples\n    with pytest.raises(ValueError):\n        shard(samples, 2, 2)\n\n\ndef test_dataset_pairs_gt_and_seed(tmp_path):\n    for category in ("text", "nontext"):\n        for kind in ("img", "gt"):\n            (tmp_path / category / kind).mkdir(parents=True)\n        for i in range(5):\n            stem = f"Black_Ink__2p0pct__{i}"\n            Image.new("RGB", (10, 10)).save(tmp_path / category / "img" / f"{stem}.jpg")\n            (tmp_path / category / "gt" / f"{stem}.txt").write_text("word\\t" + "\\t".join(["0"] * 10))\n    samples = collect(tmp_path, 2, 42)\n    assert len(samples) == 4\n    assert samples == collect(tmp_path, 2, 42)\n    assert len({s["uid"] for s in samples}) == 4\n    assert samples[0]["density"] == "2p0pct"\n    assert read_gt(samples[0]["gt_path"]) == "word"\n    original = samples[0]\n    from pathlib import Path\n    Path(original["gt_path"]).write_text("changed\\t" + "\\t".join(["0"] * 10))\n    updated = next(s for s in collect(tmp_path, 2, 42) if s["uid"] == original["uid"])\n    assert updated["gt_sha"] != original["gt_sha"]\n    assert updated["image_sha"] == original["image_sha"]\n    with pytest.raises(ValueError, match="Full test requires"):\n        collect(tmp_path)\n    (tmp_path / "text" / "gt" / "extra.txt").write_text("")\n    with pytest.raises(ValueError):\n        collect(tmp_path)\n\n\ndef test_archive_paths_flattened_and_filtered(tmp_path):\n    archive = tmp_path / "data.tar.gz"\n    with tarfile.open(archive, "w:gz") as tar:\n        for name in ("../../escape.txt", "folder/keep.txt", "ignore.jpg"):\n            info = tarfile.TarInfo(name)\n            info.size = 3\n            tar.addfile(info, io.BytesIO(b"abc"))\n    dest = tmp_path / "out"\n    extract_flat(archive, dest, ".txt")\n    assert sorted(p.name for p in dest.iterdir()) == ["escape.txt", "keep.txt"]\n    assert not (tmp_path / "escape.txt").exists()\n\n\ndef test_archive_duplicate_rejected(tmp_path):\n    archive = tmp_path / "data.tar.gz"\n    with tarfile.open(archive, "w:gz") as tar:\n        for name in ("a/x.txt", "b/x.txt"):\n            info = tarfile.TarInfo(name)\n            info.size = 1\n            tar.addfile(info, io.BytesIO(b"a"))\n    with pytest.raises(ValueError):\n        extract_flat(archive, tmp_path / "out", ".txt")\n\n\ndef test_resume_recovers_partial_last_line(tmp_path):\n    path = tmp_path / "rank0.jsonl"\n    path.write_text(\'{"uid":"a","status":"ok"}\\n{"uid":\')\n    rows = read_rows(path)\n    assert len(rows) == 1\n    rewrite_rows(path, rows)\n    assert json.loads(path.read_text())["uid"] == "a"\n    assert fingerprint({"a": 1, "b": 2}) == fingerprint({"b": 2, "a": 1})\n    assert fingerprint({"a": 1}) != fingerprint({"a": 2})\n\n\ndef test_report_marks_incomplete(tmp_path):\n    from oprb_lab.report import summarize\n    (tmp_path / "manifest.json").write_text(json.dumps([{"uid": "a"}, {"uid": "b"}]))\n    row = dict(uid="a", status="ok", category="text", occlusion_type="Ink", density="2",\n               possibly_truncated=None, latency_sec=1, **score("cat", "cat"))\n    rewrite_rows(tmp_path / "rank0.jsonl", [row])\n    result = summarize(tmp_path)\n    assert result["missing"] == 1\n    assert not result["complete"]\n    assert result["truncation_unknown"] == 1\n    assert (tmp_path / "per_image_results.csv").exists()\n\n\ndef test_report_when_all_workers_fail_loading(tmp_path):\n    from oprb_lab.report import summarize\n    (tmp_path / "manifest.json").write_text(json.dumps([{"uid": "a"}]))\n    result = summarize(tmp_path)\n    assert result["missing"] == 1\n    assert result["success"] == 0\n    assert not result["complete"]\n    assert (tmp_path / "summary.json").exists()\n\n\ndef test_report_counts_failed_pages(tmp_path):\n    from oprb_lab.report import summarize\n    (tmp_path / "manifest.json").write_text(json.dumps([{"uid": "a"}]))\n    rewrite_rows(tmp_path / "rank0.jsonl", [dict(uid="a", status="error", error="OOM")])\n    result = summarize(tmp_path)\n    assert result["errors"] == 1\n    assert result["missing"] == 0\n    assert not result["complete"]\n', 'tests/test_granite_batch.py': '"""Verify page/token alignment and EOS accounting without downloading a VLM."""\nimport contextlib\nimport sys\nfrom types import SimpleNamespace\n\nimport numpy as np\nfrom PIL import Image\n\nfrom oprb_lab.models import Adapter\n\n\ndef test_granite_batch_preserves_page_mapping_and_counts_eos(tmp_path, monkeypatch):\n    paths = []\n    for i in range(2):\n        path = tmp_path / f"{i}.png"\n        Image.new("RGB", (10+i, 10)).save(path)\n        paths.append(path)\n    class InputTensor:\n        shape = (2, 3)\n        def is_floating_point(self): return False\n    class Inputs(dict):\n        def to(self, device): return self\n    class Processor:\n        def apply_chat_template(self, messages, **kwargs): return "prompt"\n        def __call__(self, **kwargs):\n            assert len(kwargs["images"]) == 2 and kwargs["padding"]\n            return Inputs(input_ids=InputTensor())\n        def decode(self, ids, **kwargs): return ",".join(map(str, ids.tolist()))\n    class Model:\n        generation_config = SimpleNamespace(eos_token_id=9)\n        def generate(self, **kwargs):\n            assert kwargs["do_sample"] is False\n            return np.array([[1, 2, 3, 4, 5, 9, 0], [1, 2, 3, 6, 9, 0, 0]])\n    class Doc:\n        @staticmethod\n        def load_from_doctags(value, **kwargs):\n            return SimpleNamespace(export_to_text=lambda: value)\n    class Tags:\n        @staticmethod\n        def from_doctags_and_image_pairs(raw, images):\n            return f"{images[0].width}:{raw[0]}"\n    monkeypatch.setitem(sys.modules, "docling_core.types.doc", SimpleNamespace(DoclingDocument=Doc))\n    monkeypatch.setitem(sys.modules, "docling_core.types.doc.document", SimpleNamespace(DocTagsDocument=Tags))\n    a = Adapter.__new__(Adapter)\n    a.name, a.max_tokens, a.device, a.dtype = "granite", 4, "cuda:0", "fp16"\n    a.processor, a.model = Processor(), Model()\n    a.torch = SimpleNamespace(inference_mode=contextlib.nullcontext)\n    result = a.predict_batch(paths)\n    assert [r["prediction"] for r in result] == ["10:4,5,9", "11:6,9"]\n    assert [r["generated_tokens"] for r in result] == [3, 2]\n    assert not any(r["possibly_truncated"] for r in result)\n', 'tests/test_launcher.py': 'from types import SimpleNamespace\n\nimport pytest\n\nfrom oprb_lab.__main__ import show_worker_logs\n\n\ndef test_launcher_preserves_driver_paths_for_workers(tmp_path, monkeypatch):\n    from oprb_lab import __main__ as cli\n    monkeypatch.setenv("LD_LIBRARY_PATH", "/runtime/driver")\n    monkeypatch.setattr(cli, "collect", lambda *args: [])\n    monkeypatch.setattr(cli, "summarize", lambda root: {"complete": False})\n    environments = []\n    def spawn(command, **kwargs):\n        environments.append(kwargs["env"])\n        return SimpleNamespace(returncode=1, poll=lambda: 1, wait=lambda: 1)\n    monkeypatch.setattr(cli.subprocess, "Popen", spawn)\n    args = SimpleNamespace(data=tmp_path, limit=5, seed=42, gpus="0,1", model="paddle",\n                           revision="main", max_tokens=10, max_pixels=100, max_tiles=1, output=tmp_path)\n    with pytest.raises(SystemExit, match="Incomplete"):\n        cli.launch(args)\n    assert [env["CUDA_VISIBLE_DEVICES"] for env in environments] == ["0", "1"]\n    assert all(env["LD_LIBRARY_PATH"] == "/runtime/driver" for env in environments)\n\n\ndef test_failed_worker_log_tail_is_visible(tmp_path, capsys):\n    (tmp_path / "rank0.log").write_text("old\\nloading\\nRuntimeError: CUDA startup failed\\n")\n    show_worker_logs(tmp_path, [SimpleNamespace(returncode=1)], tail=2)\n    output = capsys.readouterr().out\n    assert "exit code 1" in output\n    assert "CUDA startup failed" in output\n    assert "old\\n" not in output\n\n\ndef test_worker_missing_log_still_reports_exit_code(tmp_path, capsys):\n    show_worker_logs(tmp_path, [SimpleNamespace(returncode=-9)])\n    assert "exit code -9" in capsys.readouterr().out\n', 'tests/test_notebook.py': '"""Validate generated notebooks and environment routing without downloading models."""\nimport ast\nimport json\nimport os\nimport sys\nfrom pathlib import Path\nfrom types import SimpleNamespace\n\nimport pytest\n\nROOT=Path(__file__).resolve().parents[1]\n\n\ndef notebook(name=\'notebooks/OPRB_Kaggle_2GPU.ipynb\'):\n    return json.loads((ROOT/name).read_text(encoding=\'utf-8\'))\n\n\ndef context(model, calls):\n    return dict(MODEL=model,GPUS=\'0,1\',DATA=\'unused\',TESSDATA=Path(\'assets\'),REC_BATCH_SIZE=16,\n                sys=sys,prepare_model_python=lambda:\'/isolated/python\',run=lambda *args:calls.append(args))\n\n\ndef test_five_notebooks_and_suite_embed_current_sources():\n    from generate_notebooks import NAMES\n    assert set(NAMES)=={\'paddle\',\'rapid\',\'easyocr\',\'tesseract\',\'granite\'}\n    for name in [*NAMES.values(),\'notebooks/OPRB_Kaggle_2GPU.ipynb\',\'notebooks/OPRB_Kaggle_All5_Smoke.ipynb\',\'OPRB_06_Comparative_Analysis_and_Report.ipynb\']:\n        nb=notebook(name)\n        for cell in nb[\'cells\']:\n            if cell[\'cell_type\']==\'code\': ast.parse(\'\'.join(cell[\'source\']))\n        tree=ast.parse(\'\'.join(nb[\'cells\'][1][\'source\']))\n        files=next(ast.literal_eval(n.value) for n in tree.body if isinstance(n,ast.Assign) and any(isinstance(t,ast.Name) and t.id==\'FILES\' for t in n.targets))\n        for folder in (\'src\',\'tests\',\'requirements\'):\n            for path in (ROOT/folder).rglob(\'*.py\' if folder!=\'requirements\' else \'*.txt\'):\n                assert files[path.relative_to(ROOT).as_posix()]==path.read_text(encoding=\'utf-8-sig\')\n\n\ndef test_model_isolation_keeps_driver_paths(tmp_path):\n    tree=ast.parse(\'\'.join(notebook()[\'cells\'][1][\'source\']))\n    functions=[n for n in tree.body if isinstance(n,ast.FunctionDef)]\n    calls=[]\n    env={\'LD_LIBRARY_PATH\':\'/driver\',\'PYTHONPATH\':\'/host\'}\n    ns=dict(BASE=tmp_path,ROOT=tmp_path,MODEL=\'rapid\',os=os,sys=sys,ENV=env,\n            subprocess=SimpleNamespace(run=lambda *args,**kw:calls.append((args,kw))))\n    exec(compile(ast.Module(body=functions,type_ignores=[]),\'setup\',\'exec\'),ns)\n    ns[\'PYTHON\']=ns[\'prepare_model_python\']()\n    ns[\'run\'](\'-c\',\'pass\')\n    assert \'rapid\' in calls[-1][0][0][0]\n    assert env[\'LD_LIBRARY_PATH\']==\'/driver\'\n    assert \'PYTHONPATH\' not in env\n    assert \'--system-site-packages\' not in calls[1][0][0]\n\n\n@pytest.mark.parametrize(\'model\',[\'paddle\',\'rapid\',\'easyocr\',\'granite\'])\ndef test_install_selects_matching_engine_and_complete_dependencies(model):\n    calls=[]\n    exec(\'\'.join(notebook()[\'cells\'][2][\'source\']),context(model,calls))\n    installs=[c for c in calls if c[:3]==(\'-m\',\'pip\',\'install\')]\n    editable=next(c for c in installs if \'-e\' in c)\n    assert \'.[test]\' in editable and \'--no-deps\' not in editable\n    assert any(f\'requirements/{model}.txt\' in c for c in installs)\n    if model==\'paddle\':\n        assert any(\'https://download.pytorch.org/whl/cpu\' in c for c in installs)\n        assert any(\'from paddleocr import PaddleOCR\' in str(c) for c in calls)\n    if model==\'rapid\':\n        assert (\'-m\',\'pip\',\'uninstall\',\'-y\',\'onnxruntime\') in calls\n        assert any(\'onnxruntime-gpu[cuda,cudnn]==1.23.2\' in c for c in installs)\n\n\ndef test_no_gpu_is_detected_before_querying_device_name(monkeypatch):\n    calls=[]\n    exec(\'\'.join(notebook()[\'cells\'][3][\'source\']),context(\'granite\',calls))\n    fake=SimpleNamespace(__version__=\'test\',cuda=SimpleNamespace(device_count=lambda:0,is_available=lambda:False))\n    monkeypatch.setitem(sys.modules,\'torch\',fake)\n    with pytest.raises(AssertionError,match=\'CUDA unavailable\'):\n        exec(calls[0][1])\n', 'tests/test_paddle_worker.py': '"""Paddle must run even when the installed PyTorch CUDA build cannot import."""\nimport builtins\nimport json\nimport sys\nfrom types import SimpleNamespace\n\nimport pytest\n\nfrom oprb_lab import worker\nfrom oprb_lab.storage import read_rows\n\n\n@pytest.mark.parametrize("fail_page", [False, True])\ndef test_paddle_worker_never_imports_torch(tmp_path, monkeypatch, fail_page):\n    calls = []\n    class OCR:\n        def __init__(self, **kwargs):\n            assert kwargs["text_detection_model_name"] == "PP-OCRv5_mobile_det"\n        def predict(self, path):\n            calls.append(path)\n            if fail_page and len(calls) > 1:\n                raise RuntimeError("page failed")\n            return [{"rec_texts": ["cat"]}]\n    monkeypatch.setitem(sys.modules, "paddleocr", SimpleNamespace(PaddleOCR=OCR))\n    monkeypatch.setitem(sys.modules, "paddle", SimpleNamespace(\n        seed=lambda seed: None, __version__="test", version=SimpleNamespace(cuda=lambda: "12.6"),\n        device=SimpleNamespace(cuda=SimpleNamespace(synchronize=lambda: None, get_device_name=lambda n: "fake GPU"))))\n    monkeypatch.setattr(worker, "version", lambda name: "test")\n    original_import = builtins.__import__\n    def guarded_import(name, *args, **kwargs):\n        if name.split(".")[0] == "torch":\n            raise AssertionError("Paddle tried to import broken torch")\n        return original_import(name, *args, **kwargs)\n    monkeypatch.setattr(builtins, "__import__", guarded_import)\n    (tmp_path / "config.json").write_text(json.dumps(dict(model="paddle", seed=42, revision="test", max_tokens=10, max_pixels=100, max_tiles=1)))\n    gt = tmp_path / "gt.txt"\n    gt.write_text("cat\\t" + "\\t".join(["0"] * 10))\n    (tmp_path / "manifest.json").write_text(json.dumps([dict(uid="text/a", image_path="fake.jpg", gt_path=str(gt))]))\n    worker.run(tmp_path, 0, 1)\n    rows = read_rows(tmp_path / "rank0.jsonl")\n    assert rows[0]["status"] == ("error" if fail_page else "ok")\n    if not fail_page:\n        assert rows[0]["cer"] == 0\n    assert json.loads((tmp_path / "rank0_metadata.json").read_text())["paddle"] == "test"\n', 'tests/test_worker.py': "import json\nfrom types import SimpleNamespace\n\nimport pytest\n\nfrom oprb_lab import worker\nfrom oprb_lab.models import output_text\nfrom oprb_lab.storage import read_rows\n\n\n@pytest.mark.parametrize('fail',[False,True])\ndef test_batch_worker_remainder_errors_and_resume(tmp_path,monkeypatch,fail):\n    monkeypatch.setattr(worker,'TORCH_MODELS',set())\n    calls=[]\n    class Fake:\n        parameter_count=None\n        metadata={}\n        def __init__(self,*args,**kw): pass\n        def predict(self,path): return output_text('cat')\n        def predict_batch(self,paths):\n            calls.append(paths)\n            if fail: raise RuntimeError('batch failed')\n            return [output_text('cat') for p in paths]\n    monkeypatch.setattr(worker,'Adapter',Fake)\n    gt=tmp_path/'gt.txt'\n    gt.write_text('cat\\t'+'\\t'.join(['0']*10))\n    (tmp_path/'config.json').write_text(json.dumps(dict(model='granite',seed=42,revision='test',max_tokens=10,page_batch_size=2)))\n    (tmp_path/'manifest.json').write_text(json.dumps([dict(uid=f'text/{i}',image_path=str(i),gt_path=str(gt)) for i in range(3)]))\n    worker.run(tmp_path,0,1)\n    rows=read_rows(tmp_path/'rank0.jsonl')\n    assert len(rows)==3\n    assert calls==[['0','1'],['2']]\n    assert {r['status'] for r in rows}==({'error'} if fail else {'ok'})\n    if not fail:\n        assert [r['actual_page_batch'] for r in rows]==[2,2,1]\n        assert rows[0]['latency_sec']*2==pytest.approx(rows[0]['batch_wall_sec'])\n        worker.run(tmp_path,0,1)\n        assert len(calls)==2\n\n\ndef test_tesseract_worker_does_not_import_gpu_frameworks(tmp_path,monkeypatch):\n    import builtins\n    original=builtins.__import__\n    def guard(name,*args,**kw):\n        if name in ('torch','paddle','onnxruntime'): raise AssertionError('CPU worker imported GPU framework')\n        return original(name,*args,**kw)\n    monkeypatch.setattr(builtins,'__import__',guard)\n    monkeypatch.setattr(worker,'Adapter',lambda *args,**kw:SimpleNamespace(parameter_count=None,metadata={},predict=lambda p:output_text('cat')))\n    gt=tmp_path/'gt.txt'\n    gt.write_text('cat\\t'+'\\t'.join(['0']*10))\n    (tmp_path/'config.json').write_text(json.dumps(dict(model='tesseract',seed=42,revision='x',max_tokens=1)))\n    (tmp_path/'manifest.json').write_text(json.dumps([dict(uid='text/a',image_path='a',gt_path=str(gt))]))\n    worker.run(tmp_path,0,1)\n    assert read_rows(tmp_path/'rank0.jsonl')[0]['cer']==0\n", 'requirements/easyocr.txt': 'easyocr==1.7.2\n', 'requirements/granite.txt': 'transformers==4.57.1\naccelerate>=1,<2\ndocling-core>=2.48,<3\n', 'requirements/paddle.txt': '# Install the CUDA PaddlePaddle wheel separately for your Kaggle CUDA environment.\npaddleocr==3.2.0\n', 'requirements/rapid.txt': 'rapidocr-onnxruntime==1.4.4\n', 'requirements/tesseract.txt': '# Tesseract system binary is installed separately. Python uses subprocess only.\n'}
BASE = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path("/content") if Path("/content").exists() else Path.cwd()
MODEL = 'rapid'
GPUS = "0"

ROOT = BASE / "oprb-lab"
ROOT.mkdir(parents=True, exist_ok=True)
for name, content in FILES.items():
    path = ROOT / name
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(content, encoding="utf-8")
LIMIT = 5  # None = complete 9,000-page test set, after all smoke tests pass.
DATA = Path("/tmp/oprb-data") if os.name != "nt" else BASE / "oprb-data"
OUTPUT = BASE / "oprb-results"
OUTPUT.mkdir(parents=True, exist_ok=True)
REC_BATCH_SIZE = 16  # Line crops, not pages (Paddle / RapidOCR / EasyOCR).
PAGE_BATCH_SIZE = 1  # Granite only; >1 measures amortized batch throughput.
CPU_WORKERS = 2  # Tesseract is CPU-only.
TESSDATA = OUTPUT / "_assets" / "tessdata_fast"
ENV = dict(os.environ)
ENV["MPLBACKEND"] = "Agg"
PYTHON = sys.executable

def run(*args):
    subprocess.run([PYTHON, *map(str, args)], cwd=ROOT, env=ENV, check=True)

def prepare_model_python():
    for key in ("PYTHONPATH", "PYTHONHOME", "PYTHONUSERBASE"):
        ENV.pop(key, None)
    ENV["PYTHONNOUSERSITE"] = "1"
    # Keep large virtualenvs out of Kaggle output uploads.
    target = (Path("/tmp/oprb-envs-v2") if os.name != "nt" else BASE / "oprb-envs-v2") / MODEL
    subprocess.run([sys.executable, "-m", "pip", "install", "virtualenv>=20.26"], env=ENV, check=True)
    subprocess.run([sys.executable, "-m", "virtualenv", str(target)], env=ENV, check=True)
    return str(target / ("Scripts/python.exe" if os.name == "nt" else "bin/python"))


In [ ]:

assert MODEL in ("paddle", "rapid", "easyocr", "tesseract", "granite")
gpu_ids = GPUS.split(",")
assert all(g.isdigit() for g in gpu_ids) and len(set(gpu_ids)) == len(gpu_ids), "GPUS must be unique numeric IDs"
print("Selected model:", MODEL, "GPU IDs:", GPUS, "host Python:", sys.version)
if MODEL != "tesseract":
    run("-c", "import shutil, subprocess; smi=shutil.which('nvidia-smi'); assert smi, 'Enable GPU accelerator'; subprocess.run([smi, '-L'], check=True)")
PYTHON = prepare_model_python()
if MODEL == "paddle":
    run("-m", "pip", "install", "torch==2.9.1", "--index-url", "https://download.pytorch.org/whl/cpu")
elif MODEL in ("easyocr", "granite"):
    run("-m", "pip", "install", "torch==2.9.1", "torchvision==0.24.1", "--index-url", "https://download.pytorch.org/whl/cu126")
run("-m", "pip", "install", "--upgrade-strategy", "only-if-needed", "-e", ".[test]")
run("-m", "pip", "install", "-r", f"requirements/{MODEL}.txt")
if MODEL == "paddle":
    run("-m", "pip", "install", "paddlepaddle-gpu==3.2.0", "-i", "https://www.paddlepaddle.org.cn/packages/stable/cu126/")
    run("-c", "import torch; assert torch.version.cuda is None; from paddleocr import PaddleOCR; print('PaddleOCR import passed')")
elif MODEL == "rapid":
    # The legacy wrapper depends on the CPU ORT distribution; replace that distribution
    # with the pinned GPU build, rather than installing both in the same namespace.
    run("-m", "pip", "uninstall", "-y", "onnxruntime")
    run("-m", "pip", "install", "onnxruntime-gpu[cuda,cudnn]==1.23.2")
    run("-c", "import onnxruntime as ort; ort.preload_dlls(directory=''); assert 'CUDAExecutionProvider' in ort.get_available_providers()")
elif MODEL == "tesseract":
    import shutil, urllib.request
    if not shutil.which("tesseract"):
        prefix = [] if os.geteuid() == 0 else ["sudo"]
        subprocess.run([*prefix, "apt-get", "update", "-qq"], check=True)
        subprocess.run([*prefix, "apt-get", "install", "-y", "tesseract-ocr"], check=True)
    TESSDATA.mkdir(parents=True, exist_ok=True)
    weights = TESSDATA / "eng.traineddata"
    if not weights.exists():
        # Resolve a commit before downloading pretrained tessdata_fast weights.
        request = urllib.request.Request("https://api.github.com/repos/tesseract-ocr/tessdata_fast/commits/main", headers={"User-Agent": "oprb-lab"})
        with urllib.request.urlopen(request) as response:
            revision = json.load(response)["sha"]
        temporary = weights.with_suffix(".part")
        urllib.request.urlretrieve(f"https://raw.githubusercontent.com/tesseract-ocr/tessdata_fast/{revision}/eng.traineddata", temporary)
        temporary.replace(weights)
        (TESSDATA / "revision.txt").write_text(revision)
run("-c", "import rapidfuzz, PIL, huggingface_hub, pandas, matplotlib, tqdm, pytest; from oprb_lab.metrics import score; assert score('cat','cat')['cer']==0; print('Scoring dependencies verified')")
run("-m", "pytest", "-q", "tests/test_core.py", "tests/test_comparison.py", "tests/test_adapters.py", "tests/test_launcher.py", "tests/test_worker.py")


In [ ]:

if MODEL == "paddle":
    run("-c", "import paddle; assert paddle.is_compiled_with_cuda(); "
        f"ids = {list(map(int, GPUS.split(',')))!r}; "
        "n = paddle.device.cuda.device_count(); print('Paddle:',paddle.__version__,'GPUs:',n); "
        "assert ids and min(ids)>=0 and max(ids)<n, 'Selected GPU IDs unavailable'; "
        "[print(i,paddle.device.cuda.get_device_name(i)) for i in ids]; "
        "paddle.set_device('gpu:'+str(ids[0])); paddle.zeros([1]); paddle.device.cuda.synchronize()")
elif MODEL in ("easyocr", "granite"):
    run("-c", "import torch; "
        f"ids = {list(map(int, GPUS.split(',')))!r}; "
        "n=torch.cuda.device_count(); print('Torch:',torch.__version__,'GPUs:',n); "
        "assert torch.cuda.is_available() and ids and min(ids)>=0 and max(ids)<n, 'CUDA unavailable or selected GPU IDs missing'; "
        "[print(i,torch.cuda.get_device_name(i)) for i in ids]; torch.zeros(1,device='cuda:'+str(ids[0])); torch.cuda.synchronize(ids[0])")
elif MODEL == "rapid":
    run("-c", "import onnxruntime as ort; ort.preload_dlls(directory=''); print(ort.get_available_providers())")
else:
    print("Tesseract uses CPU processes, not GPUs.")
run("-m", "oprb_lab", "download", "--data", DATA)
# Preload weights once to avoid two GPU workers racing to download the same files.
preflight = ("from huggingface_hub import model_info; from oprb_lab.models import Adapter, MODELS; "
             f"name={MODEL!r}; rev=model_info(MODELS[name]).sha if name=='granite' else 'assets'; "
             f"a=Adapter(name,rev,rec_batch_size={REC_BATCH_SIZE},tessdata={str(TESSDATA)!r}); print(a.metadata)")
run("-c", preflight)


In [ ]:

args = ["-m", "oprb_lab", "run", "--model", MODEL, "--data", DATA, "--output", OUTPUT,
        "--gpus", GPUS, "--rec-batch-size", str(REC_BATCH_SIZE), "--cpu-workers", str(CPU_WORKERS),
        "--page-batch-size", str(PAGE_BATCH_SIZE if MODEL == "granite" else 1)]
if MODEL == "tesseract":
    args += ["--tessdata", str(TESSDATA)]
if LIMIT is not None:
    args += ["--limit", str(LIMIT)]
run(*args)


In [ ]:

import shutil
for path in sorted(OUTPUT.glob("*/summary.json")):
    print(path.parent.name, path.read_text())
archive = shutil.make_archive(str(BASE / "oprb-results"), "zip", OUTPUT)
from IPython.display import FileLink, display
display(FileLink(archive))
